# Day 3 - Baselines B1 and B2
## Trustworthy Evidence-Grounded Framework - FEVEROUS Validation

**Objectives:**
1. Load Mistral-7B-Instruct-v0.3 in 4-bit quantization (bitsandbytes)
2. Run **Baseline B1** (LLM-only, no retrieved evidence) across all 450 claims
3. Run **Baseline B2** (Standard RAG: top-10 evidence dumped into prompt, no scoring)
4. Log all prompt/output/predicted label to `b1_outputs.csv` and `b2_outputs.csv`
5. Compute label accuracy for B1 and B2

---
**Expected runtime:** ~2-3 hours for 450 claims x 2 systems on T4 (Mistral-7B 4-bit)

> **Package versions are pinned to match the working Day 1 environment.**
> You will need a HF token with access to `mistralai/Mistral-7B-Instruct-v0.3`.

In [9]:
# Cell 1: Install — versions pinned to match Day 1 working environment
!pip install -q \
    "datasets==2.15.0" \
    "huggingface_hub==0.19.4" \
    "fsspec==2023.10.0" \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    bitsandbytes
print('Packages ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Cell 2: HF Login (paste your token when prompted)
from huggingface_hub import login
login()  # Opens interactive token input

In [1]:
# Cell 3: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/feverous-validation'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
print('Drive mounted')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [10]:
!pip install --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2" "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 411.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 396.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 382.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 424.5 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successful

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
# Cell 4: Imports
import json
import csv
import re
import time
import numpy as np
import torch
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: False


In [6]:
# Cell 5: Load frozen dataset and Day 2 retrieval results
with open(os.path.join(DATA_DIR, 'claims.json'), encoding='utf-8') as f:
    claims = json.load(f)

with open(os.path.join(DATA_DIR, 'evidence_corpus.json'), encoding='utf-8') as f:
    evidence_corpus = json.load(f)

import pandas as pd
retrieval_df = pd.read_csv(os.path.join(OUT_DIR, 'retrieval_results.csv'))

# Map claim_id -> top-10 retrieved corpus indices
claim_id_to_retrieved = {}
for _, row in retrieval_df.iterrows():
    retrieved_idxs = json.loads(row['retrieved_idxs'])[:10]  # top-10 for B2
    claim_id_to_retrieved[row['claim_id']] = retrieved_idxs

print(f'Claims: {len(claims)}, Corpus: {len(evidence_corpus)}')
print(f'Retrieval map loaded for {len(claim_id_to_retrieved)} claims')

Claims: 450, Corpus: 4571
Retrieval map loaded for 450 claims


In [6]:
!pip install -U \
    "transformers>=4.45,<5" \
    "huggingface_hub>=0.26.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.3" \
    "safetensors>=0.4.5"

  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.19.4
    Uninstalling huggingface-hub-0.19.4:
      Successfully uninstalled huggingface-hub-0.19.4
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.2
    Uninstalling transformers-4.40.2:
      Successfully uninstalled transformers-4.40.2
  Attempting uninstall: accelerate
    Found existing installation: ac

In [7]:
# Cell 6: Load Mistral-7B-Instruct-v0.3 in 4-bit
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Loading {MODEL_ID} in 4-bit ...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('Model loaded')

Loading mistralai/Mistral-7B-Instruct-v0.3 in 4-bit ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded


In [2]:
# Cell 7: Generation helpers

MAX_NEW_TOKENS = 96
MAX_INPUT_TOKENS = 4096

def extract_label(text):
    """Parse predicted label from model output (case-insensitive)."""
    text_upper = text.upper()

    if (
        'NOT ENOUGH INFO' in text_upper
        or 'NOT ENOUGH INFORMATION' in text_upper
        or 'NEI' in text_upper
    ):
        return 'NOT ENOUGH INFO'

    if 'REFUTES' in text_upper or 'REFUTED' in text_upper or 'FALSE' in text_upper:
        return 'REFUTES'

    if 'SUPPORTS' in text_upper or 'SUPPORTED' in text_upper or 'TRUE' in text_upper:
        return 'SUPPORTS'

    return 'UNKNOWN'


def generate_response(prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_len = inputs['input_ids'].shape[1]

    generated = tokenizer.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    )

    del inputs
    del output_ids

    return generated.strip()


def build_b1_prompt(claim):
    return (
        '[INST] You are a fact-checking assistant. Based ONLY on your training knowledge, '
        'classify the following claim as SUPPORTS, REFUTES, or NOT ENOUGH INFO. '
        'Then briefly justify your answer (1-2 sentences).\n\n'
        f'Claim: {claim}\n\n'
        'Respond with:\nVerdict: <SUPPORTS|REFUTES|NOT ENOUGH INFO>\n'
        'Justification: <one or two sentences> [/INST]'
    )


def build_b2_prompt(claim, evidence_texts):
    ev_block = '\n'.join(
        f'[Evidence {i+1}] {t}'
        for i, t in enumerate(evidence_texts)
    )

    return (
        '[INST] You are a fact-checking assistant. Using ONLY the evidence provided below, '
        'classify the claim as SUPPORTS, REFUTES, or NOT ENOUGH INFO. '
        'Justify your verdict in 1-2 sentences citing specific evidence.\n\n'
        f'Claim: {claim}\n\n'
        f'Evidence:\n{ev_block}\n\n'
        'Respond with:\nVerdict: <SUPPORTS|REFUTES|NOT ENOUGH INFO>\n'
        'Justification: <one or two sentences> [/INST]'
    )


print('Generation helpers defined')

Generation helpers defined


In [3]:
# Cell 8: Run Baseline B1 (LLM-only, no evidence)
print('=== Running Baseline B1 (LLM-only) ===')
b1_results = []

for i, claim in enumerate(tqdm(claims, desc='B1')):
    prompt = build_b1_prompt(claim['claim'])
    t0 = time.time()
    response = generate_response(prompt)
    latency = round(time.time() - t0, 2)
    pred_label = extract_label(response)

    b1_results.append({
        'claim_id':   claim['claim_id'],
        'claim':      claim['claim'],
        'gold_label': claim['label'],
        'pred_label': pred_label,
        'correct':    int(pred_label == claim['label']),
        'response':   response,
        'latency_s':  latency
    })

    if (i + 1) % 50 == 0:
        b1_acc = sum(r['correct'] for r in b1_results) / len(b1_results)
        unk = sum(1 for r in b1_results if r['pred_label'] == 'UNKNOWN') / len(b1_results)
        print(f'  [{i+1}/{len(claims)}] acc={b1_acc:.3f}  unknown_rate={unk:.3f}')

print(f'B1 complete. Accuracy: {sum(r["correct"] for r in b1_results)/len(b1_results):.4f}')

=== Running Baseline B1 (LLM-only) ===


NameError: name 'tqdm' is not defined

In [13]:
# Cell 9: Save B1 results
b1_path = os.path.join(OUT_DIR, 'b1_outputs.csv')
with open(b1_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(b1_results[0].keys()))
    writer.writeheader()
    writer.writerows(b1_results)
print(f'Saved: {b1_path}')

NameError: name 'b1_results' is not defined

In [9]:
# B2 prompt-length diagnostic
TOP_K_B2 = 10

b2_prompt_lengths = []

for claim in claims:
    retrieved_idxs = claim_id_to_retrieved.get(
        claim['claim_id'], []
    )[:TOP_K_B2]

    ev_texts = [
        evidence_corpus[idx]['text']
        for idx in retrieved_idxs
        if idx < len(evidence_corpus)
    ]

    prompt = build_b2_prompt(claim['claim'], ev_texts)

    n_tokens = len(tokenizer(prompt)['input_ids'])
    b2_prompt_lengths.append(n_tokens)

b2_prompt_lengths_sorted = sorted(b2_prompt_lengths)

print("=== B2 Prompt Length Diagnostic ===")
print(f"Examples:        {len(b2_prompt_lengths)}")
print(f"Min tokens:      {min(b2_prompt_lengths)}")
print(f"Mean tokens:     {sum(b2_prompt_lengths)/len(b2_prompt_lengths):.1f}")
print(f"Median tokens:   {b2_prompt_lengths_sorted[len(b2_prompt_lengths)//2]}")
print(f"95th percentile: {b2_prompt_lengths_sorted[int(0.95*len(b2_prompt_lengths))-1]}")
print(f"Max tokens:      {max(b2_prompt_lengths)}")
print(f"> 4096:          {sum(x > 4096 for x in b2_prompt_lengths)}")
print(f"> 8192:          {sum(x > 8192 for x in b2_prompt_lengths)}")

=== B2 Prompt Length Diagnostic ===
Examples:        450
Min tokens:      271
Mean tokens:     997.7
Median tokens:   813
95th percentile: 2085
Max tokens:      7584
> 4096:          2
> 8192:          0


In [10]:
# Cell 10: Run Baseline B2 (Standard RAG, top-10 evidence, no scoring)
TOP_K_B2 = 10
print(f'=== Running Baseline B2 (Standard RAG, top-{TOP_K_B2} evidence) ===')
b2_results = []

for i, claim in enumerate(tqdm(claims, desc='B2')):
    retrieved_idxs = claim_id_to_retrieved.get(claim['claim_id'], [])[:TOP_K_B2]
    ev_texts = [
        evidence_corpus[idx]['text']
        for idx in retrieved_idxs
        if idx < len(evidence_corpus)
    ]
    prompt = build_b2_prompt(claim['claim'], ev_texts)

    t0 = time.time()
    response = generate_response(prompt)
    latency = round(time.time() - t0, 2)
    pred_label = extract_label(response)

    b2_results.append({
        'claim_id':         claim['claim_id'],
        'claim':            claim['claim'],
        'gold_label':       claim['label'],
        'pred_label':       pred_label,
        'correct':          int(pred_label == claim['label']),
        'response':         response,
        'retrieved_idxs':   json.dumps(retrieved_idxs),
        'n_evidence_used':  len(ev_texts),
        'latency_s':        latency
    })

    if (i + 1) % 50 == 0:
        b2_acc = sum(r['correct'] for r in b2_results) / len(b2_results)
        unk = sum(1 for r in b2_results if r['pred_label'] == 'UNKNOWN') / len(b2_results)
        print(f'  [{i+1}/{len(claims)}] acc={b2_acc:.3f}  unknown_rate={unk:.3f}')

print(f'B2 complete. Accuracy: {sum(r["correct"] for r in b2_results)/len(b2_results):.4f}')

=== Running Baseline B2 (Standard RAG, top-10 evidence) ===


B2:   0%|          | 0/450 [00:00<?, ?it/s]

  [50/450] acc=0.440  unknown_rate=0.000
  [100/450] acc=0.440  unknown_rate=0.000
  [150/450] acc=0.473  unknown_rate=0.000
  [200/450] acc=0.470  unknown_rate=0.005
  [250/450] acc=0.452  unknown_rate=0.004
  [300/450] acc=0.450  unknown_rate=0.003
  [350/450] acc=0.446  unknown_rate=0.006
  [400/450] acc=0.450  unknown_rate=0.005
  [450/450] acc=0.438  unknown_rate=0.004
B2 complete. Accuracy: 0.4378


In [11]:
# Cell 11: Save B2 results
b2_path = os.path.join(OUT_DIR, 'b2_outputs.csv')
with open(b2_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(b2_results[0].keys()))
    writer.writeheader()
    writer.writerows(b2_results)
print(f'Saved: {b2_path}')

Saved: /content/drive/MyDrive/feverous-validation/outputs/b2_outputs.csv


In [15]:
import os

b1_path = '/content/drive/MyDrive/feverous-validation/outputs/b1_outputs.csv'
b2_path = '/content/drive/MyDrive/feverous-validation/outputs/b2_outputs.csv'

print("B1 exists:", os.path.exists(b1_path))
print("B1 size:", os.path.getsize(b1_path) if os.path.exists(b1_path) else "N/A")

print("B2 exists:", os.path.exists(b2_path))
print("B2 size:", os.path.getsize(b2_path) if os.path.exists(b2_path) else "N/A")

B1 exists: True
B1 size: 0
B2 exists: True
B2 size: 259301


In [14]:
# Cell 12: Accuracy summary
import pandas as pd
b1_path = '/content/drive/MyDrive/feverous-validation/outputs/b1_outputs.csv'
b2_path = '/content/drive/MyDrive/feverous-validation/outputs/b2_outputs.csv'

b1_df = pd.read_csv(b1_path)
b2_df = pd.read_csv(b2_path)

print('=' * 60)
print('LABEL ACCURACY SUMMARY  (B1 vs B2)')
print('=' * 60)
for system_name, df in [('B1 (LLM-only)', b1_df), ('B2 (Standard RAG)', b2_df)]:
    overall_acc = df['correct'].mean()
    unk_rate    = (df['pred_label'] == 'UNKNOWN').mean()
    print(f'\n{system_name}:  Overall = {overall_acc:.4f}  (UNKNOWN rate = {unk_rate:.4f})')
    for label in sorted(df['gold_label'].unique()):
        sub = df[df['gold_label'] == label]
        print(f'  {label:<24} {sub["correct"].mean():.4f}  (n={len(sub)})')
print('=' * 60)

EmptyDataError: No columns to parse from file

## Day 3 Complete

Files saved:
- `outputs/b1_outputs.csv` — B1 (LLM-only) results
- `outputs/b2_outputs.csv` — B2 (Standard RAG) results

> **Tip:** If you are running Day 4 immediately in the same session, keep the Mistral model in memory — it saves ~10 min reload time.

Next: Open `Day4_Evidence_Scoring.ipynb`